# Stage 6 - physical threat-sizing (integer + per-ID envelope)

How much PGD degradation survives real-frame constraints. Round each adversarial frame to legal integers (per-feature clip, so the ID is not crushed), then reject any frame whose bytes fall outside the observed per-ID envelope of benign traffic.

In [ ]:
import sys
from pathlib import Path
try:
    import adversec
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd().parent)); import adversec
import numpy as np, pandas as pd
from adversec import config
from adversec.contract import FEATURES, LABEL_COLUMN, ID_COLUMN, DATA_COLUMNS
pd.set_option('display.width', 140)

# The two datasets are treated identically: every step below runs the SAME
# code for both. The only dataset-specific code in the project is each
# dataset's loader (adversec/datasets/ciciov.py, road.py).
DATASETS = ['ciciov2024', 'road']

## Setup

In [ ]:
import torch, joblib
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from adversec.models import CNN1D, train_cnn
from adversec.experiments import attack as atk, realism
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', DEVICE)
def mf1(y, p): return f1_score(y, p, average='macro', zero_division=0)
id_idx = FEATURES.index(ID_COLUMN); data_idx = [FEATURES.index(c) for c in DATA_COLUMNS]

## Sweep: raw vs integer-rounded F1, and % rejected by the per-ID envelope
Rounding that does NOT recover F1 means the threat is real on the discrete grid; a high rejection rate means a cheap validity check blocks most naive attacks.

In [ ]:
for name in DATASETS:
    a = np.load(config.PROCESSED_DIR / f'{name}_stage2_arrays.npz')
    Xtr, ytr, Xte, yte = a['X_train'], a['y_train'], a['X_test'].astype(np.float32), a['y_test']
    classes = list(joblib.load(config.PROCESSED_DIR / f'{name}_label_encoder.joblib').classes_)
    scaler = joblib.load(config.PROCESSED_DIR / f'{name}_feature_scaler.joblib')
    cfg = config.load_dataset_config(name)
    cw = None
    if cfg.get('cnn_class_weights'):
        w = compute_class_weight('balanced', classes=np.unique(ytr), y=ytr)
        cw = torch.tensor(w, dtype=torch.float32, device=DEVICE)
    cnn = train_cnn(CNN1D(n_features=Xtr.shape[1], n_classes=len(classes)), Xtr, ytr, n_epochs=50, device=DEVICE, class_weights=cw)
    clf = atk.wrap_cnn_for_art(cnn, n_features=Xtr.shape[1], n_classes=len(classes), device=DEVICE)
    # benign envelope from benign train + test frames
    train_dup = pd.read_csv(config.PROCESSED_DIR / f'{name}_train_dup.csv')
    test_df   = pd.read_csv(config.PROCESSED_DIR / f'{name}_test.csv')
    benign = cfg['benign_label']
    benign_all = pd.concat([train_dup[train_dup[LABEL_COLUMN]==benign], test_df[test_df[LABEL_COLUMN]==benign]], ignore_index=True)
    ranges = realism.learn_observed_ranges(benign_all, ID_COLUMN, DATA_COLUMNS)
    print(f'\n=== {name}  (benign envelope: {len(ranges)} IDs) ===')
    print(f"{'eps':>6}{'raw_F1':>9}{'round_F1':>10}{'%rejected':>11}{'survivors':>11}")
    for eps in config.FGSM_EPSILONS:
        Xadv = atk.generate_pgd(clf, Xte, eps)
        f1_raw = mf1(yte, clf.predict(Xadv).argmax(1))
        Xround, Xint = realism.round_to_integer_frames(Xadv, scaler)
        f1_round = mf1(yte, clf.predict(Xround).argmax(1))
        mask = realism.observed_range_mask(Xint, ranges, id_idx, data_idx)
        print(f'{eps:>6.2f}{f1_raw:>9.3f}{f1_round:>10.3f}{100*(~mask).sum()/len(mask):>10.1f}%{int(mask.sum()):>11}')